# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# reproduction

## load data

In [7]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/paper_reproduct'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'pred_len', 'data_id', 'learning_rate', 'batch_size', 'lradj', 'train_epochs', 'patience']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)
df.head(4)

,model,pred_len,data_id,learning_rate,batch_size,lradj,train_epochs,patience,mse,mae
96,FBM_L,96,ETTh1,0.00002,128,None,100,5,0.380879,0.393170
38,FBM_L,192,ETTh1,0.00002,128,None,100,5,0.433798,0.421672
67,FBM_L,336,ETTh1,0.00002,128,None,100,5,0.471837,0.437959
89,FBM_L,720,ETTh1,0.00002,128,None,100,5,0.454312,0.453575


## analysis

In [10]:
df2 = df.copy()

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'Weather']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['FBM_L', 'SimpleTM', 'Fredformer', 'TimeKAN']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)
df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)
df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df2 = df2[['model', 'data_id', 'pred_len', 'mse', 'mae']]


df2 = df2.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df2 = df2[columns]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
os.makedirs(save_root, exist_ok=True)

df2.to_excel(f'{save_root}/paper_reproduct.xlsx')
df2

/tmp/ipykernel_3178761/3436697107.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model                FBM_L            SimpleTM           Fredformer            \
                       mse       mae       mse       mae        mse       mae   
data_id pred_len                                                                
ETTm1   96        0.352988  0.373396  0.332267  0.367620   0.332461  0.364627   
        192       0.390640  0.391727  0.361307  0.381322   0.361512  0.380051   
        336       0.422445  0.412836  0.391413  0.404884   0.397117  0.408334   
        720       0.482996  0.446803  0.455285  0.438535   0.456645  0.442678   
        Avg       0.412267  0.406190  0.385068  0.398091   0.386934  0.398923   
ETTm2   96        0.182027  0.264743  0.175805  0.259442   0.178983  0.262104   
        192       0.246788  0.304501  0.240863  0.300250   0.244114  0.303576   
        336       0.307263  0.342629  0.302107  0.340470   0.300989  0.339822   
        720       0.405180  0.396999  0.394534  0.396739   0.395878  0.396202   
        Avg       0.285314  0.327218  0.278327  0.324225   0.279991  0.325426   
ETTh1   96        0.380879  0.393170  0.364355  0.391751   0.376360  0.393351   
        192       0.433798  0.421672  0.423304  0.422994   0.438542  0.425398   
        336       0.471837  0.437959  0.451339  0.447843   0.478875  0.442594   
        720       0.454312  0.453575  0.467977  0.465649   0.483498  0.465534   
        Avg       0.435207  0.426594  0.426744  0.432059   0.444319  0.431719   
ETTh2   96        0.289751  0.337640  0.280951  0.338021   0.293395  0.342311   
        192       0.373539  0.388901  0.354659  0.385169   0.371592  0.390118   
        336       0.375856  0.404035  0.371774  0.404328   0.381772  0.408567   
        720       0.404285  0.427392  0.414704  0.435989   0.419047  0.440014   
        Avg       0.360858  0.389492  0.355522  0.390877   0.366451  0.395253   
Weather 96        0.196421  0.235420  0.158035  0.203866   0.176460  0.216130   
        192       0.242501  0.271877  0.205018  0.247984   0.222654  0.256765   
        336       0.291805  0.306851  0.263656  0.290267   0.280024  0.298798   
        720       0.361387  0.352971  0.339682  0.341173   0.353293  0.348154   
        Avg       0.273028  0.291780  0.241598  0.270823   0.258108  0.279962   

model              TimeKAN            
                       mse       mae  
data_id pred_len                      
ETTm1   96        0.323639  0.364235  
        192       0.357653  0.382846  
        336       0.383279  0.401415  
        720       0.447321  0.437117  
        Avg       0.377973  0.396403  
ETTm2   96        0.175149  0.257888  
        192       0.239783  0.300457  
        336       0.304619  0.345511  
        720       0.402811  0.405704  
        Avg       0.280591  0.327390  
ETTh1   96        0.371043  0.398095  
        192       0.420201  0.423672  
        336       0.446254  0.436428  
        720       0.465284  0.465205  
        Avg       0.425696  0.430850  
ETTh2   96        0.296334  0.344265  
        192       0.379348  0.394865  
        336       0.433453  0.439016  
        720       0.461846  0.460826  
        Avg       0.392745  0.409743  
Weather 96        0.163622  0.210260  
        192       0.208106  0.249647  
        336       0.262883  0.290773  
        720       0.339211  0.343271  
        Avg       0.243456  0.273488